<a href="https://colab.research.google.com/github/rouuuuuuu/wie-act/blob/main/ALBERT_V2%2BMLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install scikit-learn
!pip install pandas numpy tqdm
!pip install torch
!pip install transformers scikit-learn pandas numpy torch tqdm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 73.0 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AlbertTokenizer, AlbertModel, get_scheduler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tqdm import tqdm
from torch.optim import AdamW

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load data
df = pd.read_csv('/content/df_small_balanced_9000.csv')

# Define columns
text_column = 'CommentText'
num_columns = ['Likes', 'Replies']
label_column = 'Sentiment'

# Fill missing values
df[text_column] = df[text_column].fillna("")
df[num_columns] = df[num_columns].fillna(0)

# Encode labels
le = LabelEncoder()
df['label'] = le.fit_transform(df[label_column])

# Scale numeric features
scaler = StandardScaler()
df[num_columns] = scaler.fit_transform(df[num_columns])

# Train-test split
train_texts, test_texts, train_nums, test_nums, train_labels, test_labels = train_test_split(
    df[text_column].tolist(),
    df[num_columns].values,
    df['label'].values,
    test_size=0.2,
    random_state=42
)

# Tokenizer
tokenizer = AlbertTokenizer.from_pretrained('albert-base-v2')

# Dataset class
class CommentDataset(Dataset):
    def __init__(self, texts, nums, labels, tokenizer, max_len=128):
        self.texts = texts
        self.nums = nums
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        num = self.nums[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'num_feats': torch.tensor(num, dtype=torch.float),
            'label': torch.tensor(label, dtype=torch.long)
        }

# Data loaders
train_dataset = CommentDataset(train_texts, train_nums, train_labels, tokenizer)
test_dataset = CommentDataset(test_texts, test_nums, test_labels, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Model definition using ALBERT + MLP
class AlbertMLPClassifier(nn.Module):
    def __init__(self, num_numerical_feats, num_classes):
        super(AlbertMLPClassifier, self).__init__()
        self.albert = AlbertModel.from_pretrained('albert-base-v2')
        hidden_size = self.albert.config.hidden_size

        self.mlp = nn.Sequential(
            nn.Linear(hidden_size + num_numerical_feats, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, input_ids, attention_mask, num_feats):
        outputs = self.albert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # [CLS] token

        combined = torch.cat((cls_output, num_feats), dim=1)
        logits = self.mlp(combined)
        return logits

# Initialize model
num_classes = len(le.classes_)
model = AlbertMLPClassifier(num_numerical_feats=len(num_columns), num_classes=num_classes).to(device)

# Optimizer & scheduler
optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 30
num_training_steps = num_epochs * len(train_loader)
lr_scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
criterion = nn.CrossEntropyLoss()

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        num_feats = batch['num_feats'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask, num_feats)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        lr_scheduler.step()

        total_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    print(f"\nEpoch {epoch+1} finished. Avg Train Loss: {total_loss/len(train_loader):.4f}")

    # Evaluation
    model.eval()
    all_preds, all_probs, all_labels = [], [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            num_feats = batch['num_feats'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask, num_feats)
            probs = nn.functional.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            if probs.shape[1] == 2:
                all_probs.extend(probs[:, 1].cpu().numpy())
            else:
                all_probs.extend(probs.cpu().numpy())

    # Metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='weighted')
    recall = recall_score(all_labels, all_preds, average='weighted')
    f1 = f1_score(all_labels, all_preds, average='weighted')

    try:
        if probs.shape[1] == 2:
            auc = roc_auc_score(all_labels, all_probs)
        else:
            auc = roc_auc_score(all_labels, all_probs, multi_class='ovr')
    except:
        auc = float('nan')

    print(f"Eval Epoch {epoch+1} | Accuracy: {accuracy:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}\n")


Using device: cuda


Epoch 1: 100%|██████████| 450/450 [02:40<00:00,  2.80it/s, loss=0.532]



Epoch 1 finished. Avg Train Loss: 0.6147
Eval Epoch 1 | Accuracy: 0.7967 | Precision: 0.7958 | Recall: 0.7967 | F1: 0.7947 | AUC: 0.9279



Epoch 2: 100%|██████████| 450/450 [02:38<00:00,  2.85it/s, loss=0.54]



Epoch 2 finished. Avg Train Loss: 0.4499
Eval Epoch 2 | Accuracy: 0.7978 | Precision: 0.8001 | Recall: 0.7978 | F1: 0.7986 | AUC: 0.9263



...

Epoch 12: 100%|██████████| 450/450 [02:37<00:00,  2.85it/s, loss=0.0176]



Epoch 12 finished. Avg Train Loss: 0.0365
Eval Epoch 12 | Accuracy: 0.7956 | Precision: 0.7986 | Recall: 0.7956 | F1: 0.7968 | AUC: 0.8944



Epoch 13: 100%|██████████| 450/450 [02:38<00:00,  2.85it/s, loss=9.79e-5]



Epoch 13 finished. Avg Train Loss: 0.0261
Eval Epoch 13 | Accuracy: 0.7839 | Precision: 0.7816 | Recall: 0.7839 | F1: 0.7785 | AUC: 0.8826



Epoch 14:  20%|█▉        | 88/450 [00:30<02:08,  2.83it/s, loss=4.86e-5]

In [ ]:
model.eval()
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        probs = torch.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())  # probabilité positive pour AUC

# Calcul des métriques
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average='weighted')
recall = recall_score(all_labels, all_preds, average='weighted')
f1 = f1_score(all_labels, all_preds, average='weighted')
try:
    auc = roc_auc_score(all_labels, all_probs)
except ValueError:
    auc = 'N/A (non-binaire)'

print(f"TEST | Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}, AUC: {auc}")
